# Long Short-Term Memory (LSTM)

**Domain:** Architectures  ·  **from study list**  ·  **runnable:** yes

A refresher on the gated recurrent cell that made sequence learning practical
before attention took over — and that still shows up in streaming, low-data, and
edge settings today.

## 1. What & Why

An **LSTM** is a recurrent neural network cell (Hochreiter & Schmidhuber, 1997)
designed to carry information across many time steps without the gradient
vanishing or exploding.

**The problem it solves.** A vanilla RNN updates its hidden state with
`h_t = tanh(W·[h_{t-1}, x_t])`. Backpropagating through `T` steps multiplies the
same recurrent Jacobian `T` times, so gradients shrink toward 0 (vanish) or blow
up (explode). In practice a vanilla RNN forgets anything more than ~10 steps back.

**The fix.** The LSTM adds a separate **cell state** `C_t` that is updated almost
*additively* (`C_t = f_t * C_{t-1} + i_t * c̃_t`) rather than by repeated matrix
multiplication. Three learned **gates** decide what to erase, write, and read. The
additive path is a "gradient highway": when the forget gate stays near 1, gradient
flows back through time essentially undamped.

**Reach for an LSTM when:**

- You have **sequential / time-series** data with medium-range dependencies
  (sensor streams, control signals, log/event sequences, small NLP tasks).
- Data or compute is **limited** — LSTMs train well on modest datasets where a
  Transformer would overfit or starve.
- You need a **streaming / online / constant-memory** model: an LSTM consumes one
  step at a time with O(1) state, unlike attention's O(seq) cost per token.

**Don't reach for it when** you have long contexts, abundant data, and parallel
hardware — a Transformer will almost always win on both accuracy and training
throughput because LSTMs are inherently sequential and can't be parallelized over
time.

## 2. Mental Model

Picture a **conveyor belt** running straight through time: that belt is the cell
state `C`. Information rides along it largely untouched. At each step three
**valves** (gates) act on the belt:

```
            ┌────────────────────  C_{t-1}  ───────────────────┐
            │            (×)                      (+)           │
  C_{t-1} ──┴──────────── ✕ ───────────────────── ＋ ──────────┴──▶ C_t
                          ▲                        ▲
                     forget gate              input gate × candidate
                       f_t (σ)                 i_t (σ) × c̃_t (tanh)
                          ▲                        ▲
                          └──────── [h_{t-1}, x_t] ┘
                                         │
                                    output gate o_t (σ)
                                         │
                                  h_t = o_t × tanh(C_t)  ──▶ (to next step / output)
```

- **Forget gate** `f_t` — a per-element dial in [0,1]; multiplies the belt to
  *erase* stale memory (0 = wipe, 1 = keep).
- **Input gate** `i_t` × **candidate** `c̃_t` — decides what *new* content to add.
- **Output gate** `o_t` — decides how much of the (squashed) belt to expose as the
  hidden state `h_t` that the rest of the network sees.

The key intuition: `C` is updated by **multiply-then-add**, not repeated matmul, so
memory can persist for hundreds of steps and gradients have a clean path home.

## 3. Key Concepts

The full LSTM cell, one time step (`σ` = sigmoid, `⊙` = elementwise product):

$$
\begin{aligned}
f_t &= \sigma(W_f\,[h_{t-1}, x_t] + b_f) &&\text{forget gate}\\
i_t &= \sigma(W_i\,[h_{t-1}, x_t] + b_i) &&\text{input gate}\\
\tilde{c}_t &= \tanh(W_c\,[h_{t-1}, x_t] + b_c) &&\text{candidate}\\
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{c}_t &&\text{cell-state update}\\
o_t &= \sigma(W_o\,[h_{t-1}, x_t] + b_o) &&\text{output gate}\\
h_t &= o_t \odot \tanh(C_t) &&\text{hidden state}
\end{aligned}
$$

| Term | What it is |
|---|---|
| **Cell state `C_t`** | Long-term memory; the conveyor belt updated additively. |
| **Hidden state `h_t`** | Short-term / working output, exposed to the next layer. |
| **Gate** | A sigmoid layer in [0,1] used as a soft on/off mask. |
| **Candidate `c̃_t`** | tanh-bounded proposal of new info to write to the belt. |
| **Gradient highway** | The additive `C` path that lets gradients survive BPTT. |
| **BPTT** | Backpropagation Through Time: unroll the loop and backprop. |
| **Peephole / coupled / GRU** | Common variants (see §7). |

**Parameter count.** For input size `x` and hidden size `h`, the four gate
matrices each have shape `h × (h + x)` plus a bias `h`, so the cell has
`4·(h·(h+x) + h)` parameters. Memorize the **4** — that's the LSTM tax over a
vanilla RNN.

## 4. Setup

The worked examples below build an LSTM cell from scratch in **NumPy** — no deep
learning framework needed, which is also the clearest way to re-derive the math.
The final cell *optionally* cross-checks against PyTorch's `nn.LSTM` if it happens
to be installed; it is gated so the notebook runs either way.

In [1]:
# Core example needs only NumPy. PyTorch is optional (used by the last cell).
# %pip install numpy
# %pip install torch  # optional, for the cross-check cell

import numpy as np
print("numpy", np.__version__)

numpy 2.5.0


## 5. Worked Examples

### Example 1 — An LSTM cell forward pass, from scratch

We implement the six equations above and run a tiny cell over a short sequence.
Watch the shapes: the cell maps `(x_t, h_{t-1}, C_{t-1}) → (h_t, C_t)` and we loop
it over time.

In [2]:
rng = np.random.default_rng(0)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

class LSTMCell:
    """One LSTM cell. Weights stacked as [forget, input, candidate, output]."""
    def __init__(self, input_size, hidden_size, rng):
        self.h = hidden_size
        concat = hidden_size + input_size
        # 4 gates -> stack their weight rows; small init keeps the demo stable.
        self.W = rng.standard_normal((4 * hidden_size, concat)) * 0.3
        self.b = np.zeros(4 * hidden_size)
        # Classic trick: bias the forget gate positive so it defaults to "remember".
        self.b[:hidden_size] = 1.0

    def step(self, x, h_prev, c_prev):
        z = self.W @ np.concatenate([h_prev, x]) + self.b
        H = self.h
        f = sigmoid(z[:H])              # forget gate
        i = sigmoid(z[H:2*H])           # input gate
        g = np.tanh(z[2*H:3*H])         # candidate
        o = sigmoid(z[3*H:])            # output gate
        c = f * c_prev + i * g          # additive cell-state update
        h = o * np.tanh(c)              # exposed hidden state
        return h, c, dict(f=f, i=i, o=o)

input_size, hidden_size, T = 3, 4, 5
cell = LSTMCell(input_size, hidden_size, rng)
xs = rng.standard_normal((T, input_size))

h = np.zeros(hidden_size)
c = np.zeros(hidden_size)
for t, x in enumerate(xs):
    h, c, gates = cell.step(x, h, c)
    print(f"t={t}  |h|={np.linalg.norm(h):.3f}  |C|={np.linalg.norm(c):.3f}  "
          f"mean forget={gates['f'].mean():.3f}")

n_params = 4 * (hidden_size * (hidden_size + input_size) + hidden_size)
print(f"\nparameter count = 4*(h*(h+x)+h) = {n_params}")

t=0  |h|=0.165  |C|=0.339  mean forget=0.724
t=1  |h|=0.318  |C|=0.799  mean forget=0.687
t=2  |h|=0.355  |C|=0.894  mean forget=0.702
t=3  |h|=0.258  |C|=0.561  mean forget=0.728
t=4  |h|=0.259  |C|=0.547  mean forget=0.732

parameter count = 4*(h*(h+x)+h) = 128


### Example 2 — The forget gate as a memory dial

The whole point of the cell state is controllable persistence. Here we drive the
*same* cell state with a constant input but sweep the forget gate from "wipe" to
"keep" and watch how long a stored value survives. This is the gradient-highway
intuition made concrete: `f≈1` ⇒ memory (and gradient) persists; `f≈0` ⇒ it decays
in a step or two.

In [3]:
def decay_trace(forget, steps=12, write_at_0=1.0):
    """Write a value at t=0, then carry it with a fixed forget gate (no new input)."""
    c = write_at_0
    trace = [c]
    for _ in range(steps):
        c = forget * c          # i_t * g_t = 0 after t=0: pure forget-gate decay
        trace.append(c)
    return np.array(trace)

for f in (0.3, 0.7, 0.9, 0.99):
    tr = decay_trace(f)
    # number of steps until the stored value drops below 10% of its initial size
    half = np.argmax(tr < 0.1) if (tr < 0.1).any() else len(tr)
    print(f"forget={f:<5} value@step12={tr[-1]:.4f}  steps_to_<10%={half}")

print("\nf=0.3 forgets almost immediately; f=0.99 still holds ~89% after 12 steps.")

forget=0.3   value@step12=0.0000  steps_to_<10%=2
forget=0.7   value@step12=0.0138  steps_to_<10%=7
forget=0.9   value@step12=0.2824  steps_to_<10%=13
forget=0.99  value@step12=0.8864  steps_to_<10%=13

f=0.3 forgets almost immediately; f=0.99 still holds ~89% after 12 steps.


### Example 3 — Cross-check against PyTorch (optional)

If PyTorch is installed, this confirms our mental model lines up with the real
`nn.LSTM`: same parameter-count formula, same I/O contract (`input → output, (h, C)`).
The cell is gated so the notebook still runs top-to-bottom without torch.

In [4]:
try:
    import torch
    import torch.nn as nn

    torch.manual_seed(0)
    lstm = nn.LSTM(input_size=3, hidden_size=4, batch_first=True)
    x = torch.randn(1, 5, 3)                      # (batch, time, features)
    out, (hn, cn) = lstm(x)

    print("output (all hidden states):", tuple(out.shape))
    print("final h_n:", tuple(hn.shape), " final c_n:", tuple(cn.shape))

    total = sum(p.numel() for p in lstm.parameters())
    # nn.LSTM uses TWO bias vectors per gate (ih + hh), so 4*(h*(h+x) + 2h).
    formula = 4 * (4 * (4 + 3) + 2 * 4)
    print(f"torch params={total}  formula 4*(h*(h+x)+2h)={formula}")
except ImportError:
    print("torch not installed — skipping cross-check.")
    print("Expected shapes: out=(1,5,4), h_n=(1,1,4), c_n=(1,1,4); params=224.")

output (all hidden states): (1, 5, 4)
final h_n: (1, 1, 4)  final c_n: (1, 1, 4)
torch params=144  formula 4*(h*(h+x)+2h)=144


## 6. Gotchas & Pitfalls

- **Exploding gradients still happen.** LSTMs cure *vanishing* gradients, not
  exploding ones. Always **clip gradients** (e.g. `clip_grad_norm_` to ~1–5).
- **Forget-gate bias matters.** Initialize the forget-gate bias to **+1 (or +2)** so
  the cell defaults to remembering early in training (Jozefowicz et al., 2015).
  Many frameworks don't do this automatically.
- **They're sequential — no time parallelism.** Training can't be parallelized over
  the time axis the way a Transformer can, so long sequences are slow on GPUs.
- **Mind the two `(h, C)` states.** Both must be carried/initialized; a frequent bug
  is resetting one but not the other, or forgetting to **detach** the hidden state
  between truncated-BPTT chunks (otherwise the graph grows without bound).
- **`batch_first` confusion.** PyTorch defaults to `(time, batch, feature)`; set
  `batch_first=True` for `(batch, time, feature)` or your dims silently transpose.
- **Variable-length sequences.** Don't let padding pollute the state — use
  `pack_padded_sequence` (PyTorch) or masking so the cell skips pad steps.
- **Layer norm, not batch norm.** BatchNorm interacts badly with recurrence; use
  **LayerNorm** (or a LayerNorm-LSTM) if you need normalization.

## 7. When to Use vs Alternatives

| Option | Pick it when… | Trade-off vs LSTM |
|---|---|---|
| **Vanilla RNN** | Sequences are very short (<10 steps) and you want minimal params. | Cheaper, but vanishing gradients kill long-range memory. |
| **GRU** | You want most of the LSTM benefit with fewer params/faster training. | Merges cell+hidden and uses 2 gates; usually ~ as accurate, sometimes weaker on very long dependencies. |
| **Transformer** | Long context, lots of data, parallel hardware, top accuracy. | Wins on accuracy/throughput but O(n²) attention, more data-hungry, heavier. See `transformer.ipynb`, `attention-mechanisms.ipynb`. |
| **Temporal CNN (TCN)** | Fixed-size receptive field, want full parallelism + stable training. | Parallelizable; but receptive field is bounded by depth/dilation. |
| **State-space models (Mamba/S4)** | Very long sequences, want linear-time + parallel training. | Modern recurrent alternative; see `mamba-ssm.ipynb`. |

**Rule of thumb (2026):** for new large-scale sequence/NLP work, start with a
Transformer or SSM. Keep the LSTM in your kit for **small data, streaming/online
inference, constant-memory edge deployment, and classic time-series** where its
inductive bias and tiny footprint genuinely pay off. Closely related:
`rnn.ipynb` (the cell it fixes) and `encoder-decoder.ipynb` (where LSTMs powered
the first seq2seq systems).

## 8. Resources

- **Understanding LSTM Networks** — Christopher Olah's canonical illustrated guide:
  https://colah.github.io/posts/2015-08-Understanding-LSTMs/
- **Original paper** — Hochreiter & Schmidhuber, *Long Short-Term Memory* (1997):
  https://www.bioinf.jku.at/publications/older/2604.pdf
- **PyTorch `nn.LSTM` docs** (the I/O contract and `batch_first`/packing details):
  https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html
- **An Empirical Exploration of Recurrent Network Architectures** — Jozefowicz et
  al., 2015 (the forget-bias=1 finding, GRU vs LSTM):
  https://proceedings.mlr.press/v37/jozefowicz15.pdf
- **The Unreasonable Effectiveness of RNNs** — Karpathy, intuition + char-RNN demo:
  https://karpathy.github.io/2015/05/21/rnn-effectiveness/